# Joining Tables by Columns

Often, data about the same individuals is maintained in more than one table. For example, one university office might have data about each student's time to completion of degree, while another has data about the student's tuition and financial aid.

To understand the students' experience, it may be helpful to put the two datasets together. If the data are stored in two DataFrames, each with one row per student, then we would want to combine the columns while matching rows correctly so that each student's information remains on a single row.

Let us do this in the context of a simple example, and then use the method with a larger dataset.

In [1]:
import pandas as pd
path_data = "../../../assets/data/"

import matplotlib
matplotlib.use("Agg")
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use("fivethirtyeight")
import numpy as np

The DataFrame `cones` is one we have encountered earlier. Now suppose each flavor of ice cream comes with a rating that is stored in a separate DataFrame.

In [2]:
cones = pd.DataFrame({
    "Flavor": ["strawberry", "vanilla", "chocolate", "strawberry", "chocolate"],
    "Price": [3.55, 4.75, 6.55, 5.25, 5.75]
})
cones

,Flavor,Price
0,strawberry,3.55
1,vanilla,4.75
2,chocolate,6.55
3,strawberry,5.25
4,chocolate,5.75


In [3]:
ratings = pd.DataFrame({
    "Kind": ["strawberry", "chocolate", "vanilla"],
    "Stars": [2.5, 3.5, 4.0]
})
ratings

,Kind,Stars
0,strawberry,2.5
1,chocolate,3.5
2,vanilla,4.0


Each DataFrame has a column that contains ice cream flavors: `cones` has the column `Flavor`, and `ratings` has the column `Kind`. The entries in these columns can be used to link the two DataFrames.

The pandas function `merge` creates a new DataFrame in which each cone in the `cones` DataFrame is augmented with the `Stars` information in the `ratings` DataFrame. For each cone in `cones`, `merge` finds a row in `ratings` whose `Kind` matches the cone's `Flavor`. We have to tell pandas which columns to use when matching rows.

In [4]:
rated = pd.merge(cones, ratings, left_on="Flavor", right_on="Kind")
rated

,Flavor,Price,Kind,Stars
0,strawberry,3.55,strawberry,2.5
1,vanilla,4.75,vanilla,4.0
2,chocolate,6.55,chocolate,3.5
3,strawberry,5.25,strawberry,2.5
4,chocolate,5.75,chocolate,3.5


Each cone now has not only its price but also the rating of its flavor.

In general, a merge that augments one DataFrame with information from another looks like this:

`pd.merge(df1, df2, left_on="column_in_df1", right_on="column_in_df2")`

The new DataFrame `rated` allows us to work out the price per star, which you can think of as an informal measure of value. Low values are good because they mean that you are paying less for each rating star.

In [5]:
rated.assign(**{"$/Star": rated["Price"] / rated["Stars"]}).sort_values("$/Star")

,Flavor,Price,Kind,Stars,$/Star
1,vanilla,4.75,vanilla,4.0,1.187500
0,strawberry,3.55,strawberry,2.5,1.420000
4,chocolate,5.75,chocolate,3.5,1.642857
2,chocolate,6.55,chocolate,3.5,1.871429
3,strawberry,5.25,strawberry,2.5,2.100000


Though strawberry has the lowest rating among the three flavors, the less expensive strawberry cone does well on this measure because it does not cost a lot per star.

**Side note.** Does the order in which we list the two DataFrames matter? Let's try it. As you will see, the order of the columns changes and the row ordering may change as well, but the underlying match between records is the same.

In [6]:
pd.merge(ratings, cones, left_on="Kind", right_on="Flavor")

,Kind,Stars,Flavor,Price
0,strawberry,2.5,strawberry,3.55
1,strawberry,2.5,strawberry,5.25
2,chocolate,3.5,chocolate,6.55
3,chocolate,3.5,chocolate,5.75
4,vanilla,4.0,vanilla,4.75


Also note that a default pandas merge includes only information about items that appear in both DataFrames. Let's see an example. Suppose there is a DataFrame of reviews of some ice cream cones, and we have found the average review for each flavor.

In [7]:
reviews = pd.DataFrame({
    "Flavor": ["vanilla", "chocolate", "vanilla", "chocolate"],
    "Stars": [5, 3, 5, 4]
})
reviews

,Flavor,Stars
0,vanilla,5
1,chocolate,3
2,vanilla,5
3,chocolate,4


In [8]:
average_review = reviews.groupby("Flavor", as_index=False)["Stars"].mean()
average_review

,Flavor,Stars
0,chocolate,3.5
1,vanilla,5.0


We can merge `cones` and `average_review` by specifying the columns on which to join.

In [9]:
pd.merge(cones, average_review, on="Flavor")

,Flavor,Price,Stars
0,vanilla,4.75,5.0
1,chocolate,6.55,3.5
2,chocolate,5.75,3.5


Notice how the strawberry cones have disappeared. None of the reviews are for strawberry cones, so there is nothing with which the `strawberry` rows can be matched. This behavior is the result of pandas' default **inner join**, which keeps only rows whose key values appear in both DataFrames. Whether this is desirable depends on the analysis being performed.